# 下見 3：CR-2 の代わりのベンチマークの味見（CHASE-Math と MATH Level 5・Haiku 4.5・Claude API・CPU）

**探索の下見です（ラベルなし）。** 決まりは `notes/alt_prelook_plan.md`（リクエストを送る前に書いたもの）。

- CHASE-Math（MIT）と MATH の Level 5・答えが整数の問題（MIT）から、それぞれ 48 問。CR-2 と同じ頼み方・最大 1,024 トークンで、土台の道 1 回＋引き直し 2 回ずつ解かせます（全部で 288 回）。
- 出すもの：データごとの正答率・答えの割れ方・切れた数・CR-2 の土台の道の条件を満たす割合・長さ・CR-2 の 1 問あたりの費用の見込みと、決め方による「おすすめ」。
- 問題文・応答はどこにも表示・保存しません（数とハッシュだけ）。
- 費用は **$1 前後**の見込みで、**$3** に近づいたら止まります。途中で切れても、もう一度「すべてのセルを実行」で続きから回ります。

**使い方**：CPU のランタイム。シークレットは `ANTHROPIC_API_KEY` と `GITHUB_TOKEN`。「すべてのセルを実行」で 10〜20 分くらい。結果は Drive の `DECISION_POINTS/prelook/ALT_PRELOOK_summary.json` に入ります。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。

In [ ]:
# Cell 1 — Drive, exact head, pinned SDK, key (never printed), notifications.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

drive.mount("/content/drive")
EXPECTED_HEAD = "9b620de1c36adc509aff116c61cd39fbc8858793"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")

def get_secret(*names):
    for name in names:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value and str(value).strip():
            return str(value).strip()
    return None

token = get_secret("GITHUB_TOKEN", "GH_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
assert (REPO / "notes/alt_prelook_plan.md").exists()
token = None
env.pop("GITHUB_TOKEN", None)
sys.path.insert(0, str(REPO))
from pork import alt_prelook as P
from pork import cr2 as C
from pork import cr2_run as RN
from pork import cr1_run as R1
RN.check_prompt()
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic==" + RN.SDK_VERSION], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-2000:]
import anthropic
assert anthropic.__version__ == RN.SDK_VERSION, anthropic.__version__
_key = get_secret("ANTHROPIC_API_KEY")
assert _key, "Colab のシークレットに ANTHROPIC_API_KEY を登録してね。"
client = anthropic.Anthropic(api_key=_key, max_retries=0)
del _key
OUT = Path("/content/drive/MyDrive/DECISION_POINTS/prelook")
OUT.mkdir(parents=True, exist_ok=True)
print("exact head:", head, "| prompt:", C.prompt_sha256()[:12], "| model:", C.MODEL, "| max_tokens:", C.MAX_TOKENS, "| anthropic", anthropic.__version__)

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。中身は一言だけ（データは送りません）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = '下見3 代わりのベンチマーク CHASE-Math と MATH L5（Haiku 4.5・API・CPU）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - Alt prelook CHASE MATH', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        notify('止まりました：エラーが出ています。Colab を見てください', tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

In [ ]:
# Cell 2 — pick, call (resumable), summarize.  Numbers and hashes only.
t0 = time.time()
chase_rows = P.load_chase()
math_rows = P.load_math()
pc, ic = P.pick_chase(chase_rows)
pm, im = P.pick_math(math_rows)
del chase_rows, math_rows
problems = pc + pm
print("CHASE-Math:", ic, "| MATH:", im, "| 問題の数:", len(problems), "| 呼び出しの予定:", len(problems) * (1 + P.N_SAMPLES))
HDR = {"code_head": head, "prompt_sha256": C.prompt_sha256(), "model": C.MODEL, "max_tokens": C.MAX_TOKENS,
       "sdk": anthropic.__version__, "chase": [P.CHASE_ID, P.CHASE_REV], "math": [P.MATH_ID, P.MATH_REV],
       "seeds": [P.SEED_CHASE, P.SEED_MATH], "n_per_set": P.N_PER_SET, "n_samples": P.N_SAMPLES,
       "pool": {"chase": ic, "math": im}, "problems_sha256": P.problems_sha256(problems)}
STATE = OUT / "ALT_PRELOOK_state.json"
if STATE.exists():
    st = json.loads(STATE.read_text())
    assert st["header"] == HDR, "別の実行の記録です。止めます（Claude に教えてね）。"
    print("続きから：", len(st["done"]), "問ずみ | $%.3f" % st["spent_usd"])
else:
    st = P.new_state(HDR)
_lock_save = __import__("threading").Lock()
def save():
    with _lock_save:
        R1._atomic_write(STATE, st)
RETRY_ON = (anthropic.RateLimitError, anthropic.APIConnectionError, anthropic.InternalServerError, anthropic.APITimeoutError)
n = P.run(problems, lambda p: client.messages.create(**p), st, save, retry_on=RETRY_ON)
save()
del problems, pc, pm
summary = P.summarize(st)
summary["header"] = HDR
summary["seconds"] = round(time.time() - t0, 1)
R1._atomic_write(OUT / "ALT_PRELOOK_summary.json", summary)
for name, c in summary["sets"].items():
    print(name, "| 正答率 %.2f" % c["all_accuracy"], "| q0 %.2f" % (c["q0_mean"] or 0), "| 条件を満たす %.2f" % c["cr2_baseline_pass_share"],
          "| L 中央", c["L"]["median"], "| 切れた", c["max_tokens_stops"], "| 1 問 $%.2f" % c["cr2_usd_per_item"], "| 合格" if c["candidate"] else "| 不合格", c["passes"])
print("おすすめ:", summary["decision"], "| 今回の呼び出し:", n, "| 費用 $%.3f" % st["spent_usd"])
notify('終わりました：下見 3（代わりのベンチマーク）の結果を Drive に書きました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()